In [7]:
from typing import TypedDict, Literal
from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [8]:
load_dotenv()

True

In [9]:
client = OpenAI()

In [ ]:
class AgentState(TypedDict):
    query: str
    route: str
    confidence: float
    response: str
    probabilities: dict[str, float]

In [37]:
def billing_node(state: AgentState):
    return {
        "response": "Billing agent is handling the query"
    }

In [38]:
def technical_node(state: AgentState):
    return {
        "response": "Technical agent is handling the query",
    }

In [39]:
def shipping_node(state: AgentState):
    return {
        "response": "Shipping agent is handling the query",
    }

In [40]:
def human_node(state: AgentState):
    return {
        "response": "Human agent is handling the query",
    }

In [41]:
def router_node(state: AgentState):
    decision = client.decisions.create(
        model="gpt-6-luna",
        input=state["query"],  # query is the users' query
        questions=[
            {
                "type": "choice",
                "name": "support_route",
                "instructions": (
                    "Which support team should handle this customer request?"
                    "Choose the most appropriate option from the following choices:"
                ),
                "choices": [
                    {
                        "value": "billing",
                        "description": "For payments, refunds, invoices, billing and account issues"
                    },
                    {
                        "value": "technical",
                        "description": "For bugs, product issues, feature requests, auhentication and login issues, technical support or platform issues"
                    },
                    {
                        "value": "shipping",
                        "description": "For shipping, tracking, delivery, returns, logistics or order issues"
                    },
                    {
                        "value": "other",
                        "description": "For complex issues that require human intervention or escalation or if anything is unclear"
                    }
                ]
            }
        ]
    )

    answer = decision.answers[0]
    if answer.type != "choice":
        return {
            "route": "human",
            "confidence": 0.0,
            "probabilities": {}
        }

    probabilities = {
        item.value: item.probability
        for item in answer.probabilities
    }

    route = answer.choice
    if answer.confidence < 0.75 or route == "other":
        route = "human"

    return {
        "route": route,
        "confidence": answer.confidence,
        "probabilities": probabilities
    }

In [42]:
def choose_next_node(state: AgentState):
    return state["route"]

In [43]:
builder = StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("shipping", shipping_node)
builder.add_node("human", human_node)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    choose_next_node,
    {
        "billing": "billing",
        "technical": "technical",
        "shipping": "shipping",
        "human": "human",
    }
)

for node in ["billing", "technical", "shipping", "human"]:
    builder.add_edge(node, END)

graph = builder.compile()

In [44]:
initial_state = {
    "query": "I need help with my billing"
}

In [45]:
graph.invoke(initial_state)

{'query': 'I need help with my billing',
 'route': 'billing',
 'confidence': 1.0,
 'response': 'Billing agent is handling the query',
 'probabilities': {'billing': 1.0,
  'technical': 0.0,
  'shipping': 0.0,
  'other': 0.0}}